In [1]:
import torch
x = torch.randn(3, 3)
print(x)
print(x @ x)


tensor([[-0.1702,  0.1769,  0.3393],
        [ 0.1039,  0.3921,  0.2524],
        [ 0.7282,  2.1754,  1.4706]])
tensor([[0.2944, 0.7773, 0.4858],
        [0.2068, 0.7212, 0.5054],
        [1.1730, 4.1811, 2.9589]])


In [2]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU name:", torch.cuda.get_device_name(0))


Torch version: 2.5.1
CUDA available: True
CUDA version: 11.8
GPU name: NVIDIA RTX A1000


In [3]:
x = torch.randn(5000, 5000, device="cuda")
y = torch.randn(5000, 5000, device="cuda")
z = x @ y
print(z.mean())


tensor(-0.0063, device='cuda:0')


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ----------------------------
# Device
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

# ----------------------------
# Data
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# ----------------------------
# CNN Model
# ----------------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

model = SimpleCNN().to(device)

# ----------------------------
# Loss & Optimizer
# ----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ----------------------------
# Training Loop
# ----------------------------
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if batch_idx % 50 == 0:
            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Batch [{batch_idx}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )

    print(f"Epoch {epoch+1} avg loss: {running_loss/len(train_loader):.4f}")

print("Training finished ✅")


Using device: cuda
GPU: NVIDIA RTX A1000


100%|██████████| 170M/170M [00:18<00:00, 9.03MB/s] 


Extracting ./data\cifar-10-python.tar.gz to ./data
Epoch [1/3] Batch [0/391] Loss: 2.3020
Epoch [1/3] Batch [50/391] Loss: 1.7595
Epoch [1/3] Batch [100/391] Loss: 1.7658
Epoch [1/3] Batch [150/391] Loss: 1.3185
Epoch [1/3] Batch [200/391] Loss: 1.3579
Epoch [1/3] Batch [250/391] Loss: 1.3806
Epoch [1/3] Batch [300/391] Loss: 1.1654
Epoch [1/3] Batch [350/391] Loss: 1.1583
Epoch 1 avg loss: 1.3875
Epoch [2/3] Batch [0/391] Loss: 0.9816
Epoch [2/3] Batch [50/391] Loss: 1.0007
Epoch [2/3] Batch [100/391] Loss: 1.1275
Epoch [2/3] Batch [150/391] Loss: 1.1661
Epoch [2/3] Batch [200/391] Loss: 0.9534
Epoch [2/3] Batch [250/391] Loss: 0.9454
Epoch [2/3] Batch [300/391] Loss: 0.8426
Epoch [2/3] Batch [350/391] Loss: 0.9860
Epoch 2 avg loss: 0.9994
Epoch [3/3] Batch [0/391] Loss: 0.7011
Epoch [3/3] Batch [50/391] Loss: 0.8508
Epoch [3/3] Batch [100/391] Loss: 0.9343
Epoch [3/3] Batch [150/391] Loss: 0.8320
Epoch [3/3] Batch [200/391] Loss: 0.9419
Epoch [3/3] Batch [250/391] Loss: 0.8050
Epoch 